In [7]:
import pandas as pd
import html

# Load dataset
df = pd.read_csv("final_recipes_data.csv")

def fully_unescape(x):
    if not isinstance(x, str):
        return x
    prev = None
    current = x
    while current != prev:  # keep decoding until stable
        prev = current
        current = html.unescape(prev)
    return current

# Apply to entire dataframe
df = df.applymap(fully_unescape)

# Overwrite the same file
df.to_csv("final_recipes_data_clean.csv", index=False)

print("Done! All HTML entities fully decoded.")


C:\Users\hp\AppData\Local\Temp\ipykernel_14812\2348520659.py:18: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(fully_unescape)


Done! All HTML entities fully decoded.


In [19]:
import ast
import re

# Load your CSV
df = pd.read_csv("final_recipes_data_clean.csv")

# Function to clean steps
def clean_steps(steps_str):
    if pd.isna(steps_str):
        return []
    try:
        # fix common quote issues
        s = steps_str.replace('""', '"').replace('\\"', '"').replace('“', '"').replace('”', '"')
        steps_list = ast.literal_eval(s)
        if isinstance(steps_list, str):
            return [steps_list.strip()]
        return [step.strip() for step in steps_list if step.strip()]
    except:
        # fallback: split by comma
        s = steps_str.strip()
        s = re.sub(r'^\[|\]$', '', s)  # remove brackets
        steps = [step.strip().strip('"').strip("'") for step in re.split(r'"\s*,\s*"', s) if step.strip()]
        return steps

# Apply cleaning
df['steps'] = df['steps'].apply(clean_steps)

# Optionally do the same for ingredients_cleaned if needed
def clean_ingredients(ing_str):
    if pd.isna(ing_str):
        return []
    try:
        ing_list = ast.literal_eval(ing_str)
        if isinstance(ing_list, str):
            return [ing_list.strip()]
        return [i.strip() for i in ing_list if i.strip()]
    except:
        return [ing_str.strip()]

df['ingredients_cleaned'] = df['ingredients_cleaned'].apply(clean_ingredients)

# Save back to the same CSV
df.to_csv("final_recipes_data_clean.csv", index=False)


In [ ]:
import voyageai
import numpy as np
import pandas as pd
from langchain_core.documents import Document
from langchain_community.docstore.in_memory import InMemoryDocstore


# Init Voyage client
vo = voyageai.Client()

# embeddings = VoyageAIEmbeddings(model="voyage-3.5")

In [ ]:
from langchain_community.vectorstores import FAISS
import faiss

# Read recipes
documents = []
with open("embedding_text.txt", "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        parts = [p.strip() for p in line.split("|")]
        recipe_id = parts[0].replace("ID:", "").strip()
        name = parts[1].replace("Name:", "").strip()

        page_content = f"Name: {name}. " + " ".join(parts[2:])
        metadata = {"recipe_id": recipe_id, "name": name}

        documents.append(Document(page_content=page_content, metadata=metadata))

print(f"Loaded {len(documents)} recipes")

# Figure out embedding dimension (once)
dim = len(vo.embed(["hello"], model="voyage-3.5").embeddings[0])

# Create FAISS index
index = faiss.IndexFlatL2(dim)
vector_store = FAISS(
    embedding_function=None,   # we’re embedding manually
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

Loaded 491593 recipes


`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [8]:
import numpy as np

# # Get already embedded IDs
# already_done = set(vector_store.docstore._dict.keys())
# print(f"Already in index: {len(already_done)} docs")

batch_size = 100
for i in range(0, len(documents), batch_size):
    batch_docs = documents[i:i+batch_size]
    ids = [doc.metadata["recipe_id"] for doc in batch_docs]

    # Skip if this batch is already embedded
    if all(_id in already_done for _id in ids):
        continue  

    texts = [doc.page_content for doc in batch_docs]

    # Direct Voyage embed
    response = vo.embed(texts, model="voyage-3.5")
    embeddings_list = response.embeddings  

    # Convert to numpy array
    embeddings_array = np.array(embeddings_list, dtype="float32")

    # Add vectors to FAISS
    vector_store.index.add(embeddings_array)

    # Add docs to InMemoryDocstore
    new_docs = {doc.metadata["recipe_id"]: doc for doc in batch_docs if doc.metadata["recipe_id"] not in already_done}
    vector_store.docstore.add(new_docs)

    # Map FAISS index positions → IDs
    start_pos = len(vector_store.index_to_docstore_id)
    for offset, _id in enumerate(new_docs.keys()):
        vector_store.index_to_docstore_id[start_pos + offset] = _id
    
    # Progress + checkpoint
    if i % 1000 == 0:
        print(f"✅ Done {i}/{len(documents)}")
        vector_store.save_local("faiss_index")

# Final save
vector_store.save_local("faiss_index")
print("🎉 Resume complete, all new recipes embedded & saved")


✅ Done 0/491593
✅ Done 1000/491593
✅ Done 2000/491593
✅ Done 3000/491593
✅ Done 4000/491593
✅ Done 5000/491593
✅ Done 6000/491593
✅ Done 7000/491593
✅ Done 8000/491593
✅ Done 9000/491593
✅ Done 10000/491593
✅ Done 11000/491593
✅ Done 12000/491593
✅ Done 13000/491593
✅ Done 14000/491593
✅ Done 15000/491593
✅ Done 16000/491593
✅ Done 17000/491593
✅ Done 18000/491593
✅ Done 19000/491593
✅ Done 20000/491593
✅ Done 21000/491593
✅ Done 22000/491593
✅ Done 23000/491593
✅ Done 24000/491593
✅ Done 25000/491593
✅ Done 26000/491593
✅ Done 27000/491593
✅ Done 28000/491593
✅ Done 29000/491593
✅ Done 30000/491593
✅ Done 31000/491593
✅ Done 32000/491593
✅ Done 33000/491593
✅ Done 34000/491593
✅ Done 35000/491593
✅ Done 36000/491593
✅ Done 37000/491593
✅ Done 38000/491593
✅ Done 39000/491593
✅ Done 40000/491593
✅ Done 41000/491593
✅ Done 42000/491593
✅ Done 43000/491593
✅ Done 44000/491593
✅ Done 45000/491593
✅ Done 46000/491593
✅ Done 47000/491593
✅ Done 48000/491593
✅ Done 49000/491593
✅ Done 50000/

In [1]:
import pandas as pd
import ast  # for safely evaluating string lists
from langchain_community.vectorstores import FAISS
import voyageai

# 1️⃣ Load your cleaned CSV
df = pd.read_csv("final_recipes_data_clean.csv")
df.set_index("id", inplace=True)

# 2️⃣ Initialize VoyageAI client
vo = voyageai.Client()

# 3️⃣ Load FAISS index
vector_store = FAISS.load_local("faiss_index", None, allow_dangerous_deserialization=True)

# 4️⃣ Embed your query
query = "easy and quick chicken manchurian"
query_vec = vo.embed([query], model="voyage-3.5").embeddings

# 5️⃣ Search FAISS
results = vector_store.similarity_search_by_vector(query_vec[0], k=5)

# 6️⃣ Pretty-print recipe info
for idx, r in enumerate(results, 1):
    rid = int(r.metadata["recipe_id"])
    if rid in df.index:
        recipe = df.loc[rid]
        
        # Convert string representation of lists to actual Python lists
        try:
            ingredients = ast.literal_eval(recipe["ingredients_cleaned"])
        except:
            ingredients = recipe["ingredients_cleaned"]  # fallback
        
        try:
            steps = ast.literal_eval(recipe["steps"])
        except:
            steps = recipe["steps"]  # fallback
        
        print(f"Recipe #{idx}: {recipe['name']}")
        print(f"Servings: {recipe.get('servings', 'N/A')}\n")
        
        print("Ingredients:")
        if isinstance(ingredients, list):
            for item in ingredients:
                print(f"  • {item}")
        else:
            print(f"  {ingredients}")
        
        print("\nSteps:")
        if isinstance(steps, list):
            for i, step in enumerate(steps, 1):
                print(f"  {i}. {step}")
        else:
            print(f"  {steps}")
        
        print("\n" + "="*60 + "\n")
    else:
        print(f"Recipe ID {rid} not found in CSV")


`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


Recipe #1: Chicken Manchurian
Servings: 4.0

Ingredients:
  • 2 chicken breasts, cut into small cubes
  • 1 & ½ teaspoons salt
  • 1 teaspoon ground pepper
  • 1 egg
  • 3 tablespoons cornflour
  • oil (for deep frying)
  • 2 chopped green chilies
  • 2 inches chopped ginger
  • ¼ cup soya sauce
  • ¼ cup tomato ketchup
  • 2 tablespoons oyster sauce
  • 2 tablespoons cornflour, mixed in ½ cup water

Steps:
  1. Wash chicken cubes.
  2. Drain.
  3. Mix with salt, pepper, egg and cornflour.
  4. Heat wok with oil and deep fry each piece of chicken.
  5. Remove and keep aside.
  6. In another wok, take 4 tbsp of oil.
  7. When hot add the chopped chillies, ginger and garlic.
  8. Fry for 2 minutes.
  9. Add the sauces.
  10. Stir for a minute.
  11. Add 1 cup of water and bring to a boil.
  12. Add the fried chicken pieces.
  13. If more gravy is needed, add another half cup of water.
  14. Bring it to a boil.
  15. Mix the cornflour into the chicken gravy.
  16. Stir continuously till a